In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("ayurvedic_meal_dataset.csv")
print(df.shape)
df.head()

(2500, 38)


,age,gender,disease,diet_preference,weight_kg,height_cm,bmi,bmi_category,meal_category,foods_to_avoid,...,dish_3_role,dish_3_portion_scale,dish_3_portion_pct_of_meal,dish_4,dish_4_diet_type,dish_4_role,dish_4_portion_scale,dish_4_portion_pct_of_meal,record_id,portion_pct_sum
0,51,female,arthritis,non-veg,59.7,157,24.23,normal,lunch,"Deep-fried foods, processed meats/snacks, high...",...,veg,0.870,19.11,Mango pickle (Aam ka achaar),veg,side,0.500,18.79,1,100.00
1,49,female,diabetes,non-veg,65.4,161,25.22,overweight,dinner,"Added sugar, sweets, sweetened drinks, refined...",...,veg,0.500,32.99,Vegetarian nargisi kofta curry,veg,side,0.500,23.72,2,100.00
2,33,female,gastritis,veg,66.4,158,26.60,overweight,breakfast,"Spicy foods, acidic foods (citrus/tomato), cof...",...,side,2.500,11.67,NaN,NaN,NaN,0.000,0.00,3,100.01
3,52,female,gastritis,veg,60.4,170,20.89,normal,lunch,"Spicy foods, acidic foods (citrus/tomato), cof...",...,veg,0.806,20.00,Pea vadi curry,veg,side,0.770,15.00,4,100.00
4,26,male,diabetes,veg,86.5,165,31.78,obese,lunch,"Added sugar, sweets, sweetened drinks, refined...",...,veg,0.500,35.02,Fermented bamboo shoot pickle (Mesu pickle),veg,side,1.399,13.23,5,100.00


In [6]:
X = df[['age','gender','disease','diet_preference',
        'weight_kg','height_cm','bmi',
        'bmi_category','meal_category']]

y = df['dish_1']

print(X.shape, y.shape)

(2500, 9) (2500,)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (2000, 9)
Test size: (500, 9)


In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = ['gender','disease','diet_preference','bmi_category','meal_category']
num_cols = ['age','weight_kg','height_cm','bmi']

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.034


In [10]:
df['dish_1'].nunique()

95

In [11]:
df = pd.read_csv("ayurvedic_meal_dataset.csv")

In [12]:
def check_veg_consistency(row):
    if row['diet_preference'] == 'veg':
        dishes = [row.get(f'dish_{i}', '') for i in range(1,5)]
        for d in dishes:
            if isinstance(d, str) and any(x in d.lower() for x in ['chicken','mutton','beef','pork','fish','egg']):
                return 0
    return 1

df['veg_consistency'] = df.apply(check_veg_consistency, axis=1)

veg_accuracy = df['veg_consistency'].mean()
print("Veg consistency accuracy:", veg_accuracy)

Veg consistency accuracy: 1.0


In [13]:
df['portion_pct_sum'] = (
    df['dish_1_portion_pct_of_meal'] +
    df['dish_2_portion_pct_of_meal'] +
    df['dish_3_portion_pct_of_meal'] +
    df['dish_4_portion_pct_of_meal']
)

portion_accuracy = (
    ((df['portion_pct_sum'] >= 99.5) & (df['portion_pct_sum'] <= 100.5)).mean()
)

print("Portion % correctness accuracy:", portion_accuracy)

Portion % correctness accuracy: 1.0


In [14]:
import re

disease_keywords = {
    'diabetes': ['sugar','sweet','cake','cookie','chocolate','jam','honey','cola','soda'],
    'migraine': ['chocolate','coffee','caffeine','cheese','cola'],
    'arthritis': ['fried','chips','sweet','processed'],
    'asthma': ['ice','cold','fried','processed','cheese'],
    'gastritis': ['spicy','chilli','pepper','masala','citrus','lemon','vinegar','coffee','tea','fried']
}

def disease_check(row):
    disease = row['disease']
    dishes = [row.get(f'dish_{i}', '') for i in range(1,5)]
    keywords = disease_keywords[disease]

    for dish in dishes:
        if isinstance(dish, str):
            for k in keywords:
                if re.search(k, dish.lower()):
                    return 0
    return 1

df['disease_consistency'] = df.apply(disease_check, axis=1)

disease_accuracy = df['disease_consistency'].mean()

print("Disease filtering accuracy:", disease_accuracy)

Disease filtering accuracy: 1.0


In [16]:
df['calorie_diff_pct'] = abs(
    (df['total_calories_kcal'] - df['target_meal_calories_kcal'])
    / df['target_meal_calories_kcal']
) * 100

# consider accurate if within 20% of target
calorie_accuracy_20 = (df['calorie_diff_pct'] <= 20).mean()

print("BMI calorie alignment accuracy (within 20%):", calorie_accuracy_20)
print("Average calorie deviation %:", df['calorie_diff_pct'].mean())

BMI calorie alignment accuracy (within 20%): 0.5808
Average calorie deviation %: 30.924059287089527


In [17]:
X = df[['age','gender','disease','diet_preference',
        'weight_kg','height_cm','bmi','bmi_category']]

y = df['meal_category']

print(X.shape, y.shape)

(2500, 8) (2500,)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = ['gender','disease','diet_preference','bmi_category']
num_cols = ['age','weight_kg','height_cm','bmi']

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.328


In [21]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

   breakfast       0.34      0.31      0.33       161
      dinner       0.36      0.38      0.37       171
       lunch       0.29      0.29      0.29       168

    accuracy                           0.33       500
   macro avg       0.33      0.33      0.33       500
weighted avg       0.33      0.33      0.33       500



In [22]:
X = df[['age','gender','weight_kg','height_cm']]
y = df['bmi_category']

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cat_cols = ['gender']
num_cols = ['age','weight_kg','height_cm']

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("BMI Category Prediction Accuracy:", accuracy_score(y_test, pred))

BMI Category Prediction Accuracy: 0.946


In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
f1 = f1_score(y_test, pred, average='weighted')

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Accuracy : 0.946
Precision: 0.9461
Recall   : 0.946
F1 Score : 0.946


In [24]:
def predict_bmi_category(age, gender, weight, height):
    input_df = pd.DataFrame({
        'age': [age],
        'gender': [gender],
        'weight_kg': [weight],
        'height_cm': [height]
    })

    predicted_category = rf.predict(input_df)[0]
    bmi_value = weight / ((height/100)**2)

    return predicted_category, round(bmi_value, 2)

In [27]:
def generate_meal_pretty(age, gender, weight, height, disease, meal_category, preference):
    predicted_bmi_cat, bmi_value = predict_bmi_category(age, gender, weight, height)

    filtered = df[
        (df['disease'] == disease) &
        (df['meal_category'] == meal_category) &
        (df['diet_preference'] == preference) &
        (df['bmi_category'] == predicted_bmi_cat)
    ]

    if len(filtered) == 0:
        return "No exact match found."

    result = filtered.sample(1).iloc[0]

    return {
        "User BMI": bmi_value,
        "Predicted BMI Category": predicted_bmi_cat,
        "Meal Category": meal_category,
        "Diet Preference": preference,
        "Disease": disease,
        "Foods to Avoid": result['foods_to_avoid'],

        "Meal Plan": [
            {"Dish": result['dish_1'], "Portion %": clean_number(result['dish_1_portion_pct_of_meal'])},
            {"Dish": result['dish_2'], "Portion %": clean_number(result['dish_2_portion_pct_of_meal'])},
            {"Dish": result['dish_3'], "Portion %": clean_number(result['dish_3_portion_pct_of_meal'])},
            {"Dish": result['dish_4'], "Portion %": clean_number(result['dish_4_portion_pct_of_meal'])},
        ],

        "Total Calories (kcal)": clean_number(result['total_calories_kcal']),
        "Protein (g)": clean_number(result['protein_g']),
        "Carbs (g)": clean_number(result['carbs_g']),
        "Fats (g)": clean_number(result['fats_g'])
    }

In [29]:
def clean_number(x):
    try:
        return float(x)
    except:
        return x

In [30]:
generate_meal_pretty(
    age=28,
    gender='female',
    weight=60,
    height=165,
    disease='migraine',
    meal_category='lunch',
    preference='veg'
)

{'User BMI': 22.04,
 'Predicted BMI Category': 'normal',
 'Meal Category': 'lunch',
 'Diet Preference': 'veg',
 'Disease': 'migraine',
 'Foods to Avoid': 'Caffeinated drinks, chocolate, aged/strong cheese, highly processed foods',
 'Meal Plan': [{'Dish': 'Keema parantha/paratha', 'Portion %': 35.0},
  {'Dish': 'Chickpea flour cookies (Sweet besan rounds/cookies)',
   'Portion %': 30.0},
  {'Dish': 'Russian salad', 'Portion %': 20.0},
  {'Dish': 'Pea keema curry (Matar keema ki sabzi)', 'Portion %': 15.0}],
 'Total Calories (kcal)': 648.37,
 'Protein (g)': 23.97,
 'Carbs (g)': 66.65,
 'Fats (g)': 31.28}

In [31]:
nonveg_words = ['chicken','mutton','beef','pork','fish','egg','keema','prawn','shrimp','crab']

def is_veg_safe(dish):
    if not isinstance(dish, str):
        return True
    d = dish.lower()
    return not any(w in d for w in nonveg_words)

def generate_meal_pretty_safe(age, gender, weight, height, disease, meal_category, preference):
    predicted_bmi_cat, bmi_value = predict_bmi_category(age, gender, weight, height)

    filtered = df[
        (df['disease'] == disease) &
        (df['meal_category'] == meal_category) &
        (df['diet_preference'] == preference) &
        (df['bmi_category'] == predicted_bmi_cat)
    ].copy()

    # extra safety: if veg, remove any row that contains non-veg keywords in dishes
    if preference == "veg":
        for i in range(1,5):
            filtered = filtered[filtered[f'dish_{i}'].apply(is_veg_safe)]

    if len(filtered) == 0:
        return "No exact match found after veg/non-veg safety filtering."

    result = filtered.sample(1).iloc[0]

    return {
        "User BMI": bmi_value,
        "Predicted BMI Category": predicted_bmi_cat,
        "Meal Category": meal_category,
        "Diet Preference": preference,
        "Disease": disease,
        "Foods to Avoid": result['foods_to_avoid'],

        "Meal Plan": [
            {"Dish": result['dish_1'], "Portion %": clean_number(result['dish_1_portion_pct_of_meal'])},
            {"Dish": result['dish_2'], "Portion %": clean_number(result['dish_2_portion_pct_of_meal'])},
            {"Dish": result['dish_3'], "Portion %": clean_number(result['dish_3_portion_pct_of_meal'])},
            {"Dish": result['dish_4'], "Portion %": clean_number(result['dish_4_portion_pct_of_meal'])},
        ],

        "Total Calories (kcal)": clean_number(result['total_calories_kcal']),
        "Protein (g)": clean_number(result['protein_g']),
        "Carbs (g)": clean_number(result['carbs_g']),
        "Fats (g)": clean_number(result['fats_g'])
    }

In [32]:
generate_meal_pretty_safe(
    age=28,
    gender='female',
    weight=60,
    height=165,
    disease='migraine',
    meal_category='lunch',
    preference='veg'
)

{'User BMI': 22.04,
 'Predicted BMI Category': 'normal',
 'Meal Category': 'lunch',
 'Diet Preference': 'veg',
 'Disease': 'migraine',
 'Foods to Avoid': 'Caffeinated drinks, chocolate, aged/strong cheese, highly processed foods',
 'Meal Plan': [{'Dish': 'Soya roti', 'Portion %': 28.4},
  {'Dish': 'Chilli paneer', 'Portion %': 48.23},
  {'Dish': 'Carrot halwa (Gajar ka halwa)', 'Portion %': 16.23},
  {'Dish': 'Bottle gourd soup (Ghiya/Lauki soup)', 'Portion %': 7.15}],
 'Total Calories (kcal)': 806.08,
 'Protein (g)': 11.93,
 'Carbs (g)': 50.86,
 'Fats (g)': 61.29}